# SQL for Beginners: a hands-on Colab
**Runs entirely in your browser. No installs, no uploads.** Use *Runtime > Run all*, or run cells one at a time with Shift+Enter.

We use **SQLite**, which is built into Python, so the notebook works anywhere. Commands are standard SQL, and the few differences from **MySQL** (used in MSDS 525) are noted in a table near the end.

**Topics:** DDL (CREATE, ALTER, DROP) · DML (INSERT, UPDATE, DELETE) · SELECT, WHERE, ORDER BY, LIMIT · functions (aggregate, text, number, date, NULL, CASE) · GROUP BY and HAVING · JOINs · subqueries, CTEs, views · window functions · set operations · transactions · practice exercises.

The sample data is a **small, made-up** set themed on airport traffic (incidents, weather, flights) so it connects to the BNA project. It is *not* real data.

## 0. Setup
Run cells **in order, from the top**. If you get an error such as 'UNIQUE constraint failed' or 'table already exists', you ran a cell twice: choose *Runtime > Restart and run all*. Run this cell first. It opens an in-memory database and defines a helper `sql()` that runs any statement and shows results as a table.

In [1]:
import sqlite3, pandas as pd
pd.set_option("display.width", 140)

conn = sqlite3.connect(":memory:")          # a fresh, empty database that lives in RAM
conn.execute("PRAGMA foreign_keys = ON")     # make SQLite enforce FOREIGN KEY rules

def sql(query, params=()):
    """Run one SQL statement. SELECTs return a table; other statements print rows changed."""
    cur = conn.execute(query, params)
    if cur.description:                      # has columns -> it was a SELECT
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)
    conn.commit()
    print(f"OK ({cur.rowcount} rows affected)" if cur.rowcount >= 0 else "OK")

def script(text):
    """Run several statements separated by semicolons."""
    conn.executescript(text)
    conn.commit()
    print("Script OK")

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.45.1


## 1. DDL: Data Definition Language
DDL defines the **structure** of the database: `CREATE`, `ALTER`, `DROP`.

A table has **columns** with **data types** and **constraints**:
| Constraint | Meaning |
|---|---|
| `PRIMARY KEY` | unique id for each row |
| `NOT NULL` | value required |
| `UNIQUE` | no duplicates allowed |
| `DEFAULT x` | value used when none is given |
| `CHECK (...)` | rule the value must satisfy |
| `FOREIGN KEY` | must match a row in another table |

Common types: `INTEGER`, `REAL` (decimal), `TEXT` (words, dates), `NUMERIC`. (MySQL has `INT`, `DECIMAL`, `VARCHAR`, `DATE`, `DATETIME`.)

In [2]:
script("""
DROP TABLE IF EXISTS incidents;
DROP TABLE IF EXISTS weather;
DROP TABLE IF EXISTS airport_ops;
DROP TABLE IF EXISTS road;

CREATE TABLE road (
    road_id    INTEGER PRIMARY KEY,
    road_name  TEXT NOT NULL UNIQUE,
    speed_limit INTEGER CHECK (speed_limit BETWEEN 15 AND 75)
);

CREATE TABLE incidents (
    incident_id INTEGER PRIMARY KEY,
    occurred_at TEXT NOT NULL,                -- 'YYYY-MM-DD HH:MM:SS'
    road_id     INTEGER,
    incident_type TEXT DEFAULT 'Unknown',
    injuries    INTEGER DEFAULT 0,
    FOREIGN KEY (road_id) REFERENCES road(road_id)
);

CREATE TABLE weather (
    day   TEXT PRIMARY KEY,                   -- 'YYYY-MM-DD'
    prcp  REAL,                               -- precipitation in inches
    tmax  INTEGER                             -- max temp, F
);

CREATE TABLE airport_ops (
    month      TEXT PRIMARY KEY,              -- 'YYYY-MM'
    arrivals   INTEGER,
    departures INTEGER
);
""")

Script OK


**ALTER** changes an existing table; **CREATE INDEX** speeds up searches; **PRAGMA table_info** shows a table's structure (in MySQL use `DESCRIBE table;`).

In [3]:
sql("ALTER TABLE incidents ADD COLUMN severity TEXT")          # add a column
sql("CREATE INDEX idx_incidents_time ON incidents(occurred_at)") # index for faster lookups
sql("PRAGMA table_info(incidents)")

OK
OK


,cid,name,type,notnull,dflt_value,pk
0,0,incident_id,INTEGER,0,NaN,1
1,1,occurred_at,TEXT,1,NaN,0
2,2,road_id,INTEGER,0,NaN,0
3,3,incident_type,TEXT,0,'Unknown',0
4,4,injuries,INTEGER,0,0,0
5,5,severity,TEXT,0,NaN,0


In [4]:
# DROP removes a table completely. Try it on a throwaway table.
sql("CREATE TABLE scratch (x INTEGER)")
sql("DROP TABLE scratch")
sql("SELECT name FROM sqlite_master WHERE type='table'")    # list tables (MySQL: SHOW TABLES;)

OK
OK


,name
0,road
1,incidents
2,weather
3,airport_ops


### Constraints in action
The database refuses data that breaks a rule. These cells are *supposed* to fail; we catch the error so you can read it.

In [5]:
tests = {
  "CHECK: speed limit out of range":   "INSERT INTO road (road_name, speed_limit) VALUES ('Test Rd', 200)",
  "NOT NULL: missing road_name":       "INSERT INTO road (speed_limit) VALUES (30)",
  "FOREIGN KEY: road_id 999 does not exist": "INSERT INTO incidents (incident_id, occurred_at, road_id) VALUES (1, '2026-01-01 08:00:00', 999)",
}
for label, stmt in tests.items():
    try:
        conn.execute(stmt); print("Unexpectedly allowed:", label)
    except sqlite3.IntegrityError as e:
        print(f"Blocked ({label}) ->", e)

Blocked (CHECK: speed limit out of range) -> CHECK constraint failed: speed_limit BETWEEN 15 AND 75
Blocked (NOT NULL: missing road_name) -> NOT NULL constraint failed: road.road_name
Blocked (FOREIGN KEY: road_id 999 does not exist) -> FOREIGN KEY constraint failed


## 2. DML: Data Manipulation Language
DML changes the **data**: `INSERT`, `UPDATE`, `DELETE`.

### INSERT

In [6]:
sql("INSERT INTO road (road_name, speed_limit) VALUES ('Donelson Pike', 45)")
script("""
INSERT INTO road (road_name, speed_limit) VALUES
  ('Briley Parkway', 55),
  ('Murfreesboro Pike', 40),
  ('I-40', 70);
""")
sql("SELECT * FROM road")

OK (1 rows affected)
Script OK


,road_id,road_name,speed_limit
0,1,Donelson Pike,45
1,2,Briley Parkway,55
2,3,Murfreesboro Pike,40
3,4,I-40,70


Now load many rows quickly. The data below is **generated with a fixed seed**, so everyone gets the same results. Real projects load CSVs instead (see the note after this cell).

In [7]:
import random, datetime as dt
random.seed(525)

start = dt.date(2026, 1, 1)
days = [start + dt.timedelta(days=i) for i in range(90)]          # Jan-Mar 2026

# weather: one row per day
w_rows = []
for d in days:
    rain = random.random() < 0.35
    w_rows.append((d.isoformat(), round(random.uniform(0.05, 1.2), 2) if rain else 0.0,
                   random.randint(35, 70)))
conn.executemany("INSERT INTO weather VALUES (?,?,?)", w_rows)

# incidents: more on rainy days and at 7-9 AM / 4-6 PM
types = ["Crash", "Stalled vehicle", "Debris", "Minor collision"]
wx = {r[0]: r[1] for r in w_rows}
i_rows, iid = [], 1
for d in days:
    n = random.randint(0, 2) + (3 if wx[d.isoformat()] > 0 else 0)
    for _ in range(n):
        hour = random.choice([7, 8, 9, 16, 17, 17, 18, random.randint(0, 23)])
        ts = f"{d.isoformat()} {hour:02d}:{random.randint(0,59):02d}:00"
        i_rows.append((iid, ts, random.randint(1, 4), random.choice(types), random.choice([0,0,0,1,2])))
        iid += 1
conn.executemany("INSERT INTO incidents (incident_id, occurred_at, road_id, incident_type, injuries) VALUES (?,?,?,?,?)", i_rows)

# airport operations: one row per month
conn.executemany("INSERT INTO airport_ops VALUES (?,?,?)",
                 [("2026-01", 11800, 11750), ("2026-02", 11200, 11300), ("2026-03", 13100, 13050)])
conn.commit()
print(len(w_rows), "weather rows,", len(i_rows), "incident rows")

90 weather rows, 167 incident rows


> **Loading a real CSV** (for your project): upload the file with the folder icon on the left, then
> ```python
> df = pd.read_csv("Traffic_BNA_Feb2026.csv")
> df.to_sql("raw_traffic", conn, if_exists="replace", index=False)
> ```
> In MySQL Workbench you would use the **Table Data Import Wizard** instead.

### UPDATE and DELETE
**Always use WHERE** with `UPDATE` and `DELETE`. Without it, *every* row changes. Check with a `SELECT` first. Statements run in order, so a later `UPDATE` can overwrite an earlier one (below, rows that are both 'Low' and have injuries end up 'High').

In [8]:
# 1) Preview which rows will change
print(sql("SELECT COUNT(*) AS n FROM incidents WHERE incident_type = 'Minor collision'"))
# 2) Change them
sql("UPDATE incidents SET severity = 'Low' WHERE incident_type IN ('Debris', 'Stalled vehicle')")
sql("UPDATE incidents SET severity = 'High' WHERE injuries >= 1")
sql("UPDATE incidents SET severity = 'Medium' WHERE severity IS NULL")
sql("SELECT severity, COUNT(*) AS n FROM incidents GROUP BY severity")

    n
0  41
OK (84 rows affected)
OK (73 rows affected)
OK (49 rows affected)


,severity,n
0,High,73
1,Low,45
2,Medium,49


In [9]:
# DELETE: remove rows. Here we add a bad row and then remove it.
sql("INSERT INTO incidents (incident_id, occurred_at, road_id) VALUES (9999, '2026-01-01 00:00:00', 1)")
sql("DELETE FROM incidents WHERE incident_id = 9999")
sql("SELECT COUNT(*) AS rows_left FROM incidents")

OK (1 rows affected)
OK (1 rows affected)


,rows_left
0,167


## 3. SELECT: reading data
Order of writing: `SELECT` → `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `ORDER BY` → `LIMIT`.

In [10]:
sql("SELECT * FROM incidents LIMIT 5")                                   # first 5 rows

,incident_id,occurred_at,road_id,incident_type,injuries,severity
0,1,2026-01-01 17:35:00,3,Minor collision,0,Medium
1,2,2026-01-03 16:46:00,3,Debris,2,High
2,3,2026-01-04 18:02:00,4,Minor collision,0,Medium
3,4,2026-01-04 07:00:00,2,Stalled vehicle,1,High
4,5,2026-01-04 17:10:00,1,Minor collision,2,High


In [11]:
sql("""
SELECT incident_id AS id, incident_type AS type, injuries      -- AS gives a column a nicer name
FROM incidents
WHERE injuries > 0 AND incident_type = 'Crash'
ORDER BY occurred_at DESC                                       -- newest first
LIMIT 5
""")

,id,type,injuries
0,158,Crash,2
1,151,Crash,1
2,147,Crash,1
3,144,Crash,1
4,140,Crash,2


**WHERE operators:** `=  <>  <  >  <=  >=`, `AND / OR / NOT`, `IN (...)`, `BETWEEN a AND b`, `LIKE` (pattern: `%` = any text, `_` = one character), `IS NULL`.

In [12]:
display(sql("SELECT day, prcp FROM weather WHERE prcp BETWEEN 0.5 AND 1.0 ORDER BY prcp DESC LIMIT 5"))
display(sql("SELECT DISTINCT incident_type FROM incidents"))                 # unique values
display(sql("SELECT * FROM road WHERE road_name LIKE '%Pike'"))              # ends with Pike
display(sql("SELECT * FROM incidents WHERE severity IS NULL LIMIT 3"))       # NULL needs IS NULL, not = NULL

,day,prcp
0,2026-02-21,1.00
1,2026-02-07,0.93
2,2026-02-09,0.85
3,2026-01-26,0.68
4,2026-01-21,0.64


,incident_type
0,Minor collision
1,Debris
2,Stalled vehicle
3,Crash


,road_id,road_name,speed_limit
0,1,Donelson Pike,45
1,3,Murfreesboro Pike,40


,incident_id,occurred_at,road_id,incident_type,injuries,severity


## 4. Functions
### 4a. Aggregate functions
They collapse many rows into one value: `COUNT, SUM, AVG, MIN, MAX`.

In [13]:
sql("""
SELECT COUNT(*)           AS total_incidents,
       COUNT(DISTINCT road_id) AS roads_involved,
       SUM(injuries)      AS total_injuries,
       ROUND(AVG(injuries), 2) AS avg_injuries,
       MIN(occurred_at)   AS first_seen,
       MAX(occurred_at)   AS last_seen
FROM incidents
""")

,total_incidents,roads_involved,total_injuries,avg_injuries,first_seen,last_seen
0,167,4,106,0.63,2026-01-01 17:35:00,2026-03-30 16:04:00


### 4b. Text functions
`UPPER, LOWER, LENGTH, SUBSTR, TRIM, REPLACE`, and `||` to join text (MySQL uses `CONCAT(a, b)`).

In [14]:
sql("""
SELECT road_name,
       UPPER(road_name)            AS upper_name,
       LENGTH(road_name)           AS name_length,
       SUBSTR(road_name, 1, 5)     AS first5,
       REPLACE(road_name, 'Pike', 'Pk') AS short_name,
       road_name || ' (' || speed_limit || ' mph)' AS label
FROM road
""")

,road_name,upper_name,name_length,first5,short_name,label
0,Donelson Pike,DONELSON PIKE,13,Donel,Donelson Pk,Donelson Pike (45 mph)
1,Briley Parkway,BRILEY PARKWAY,14,Brile,Briley Parkway,Briley Parkway (55 mph)
2,Murfreesboro Pike,MURFREESBORO PIKE,17,Murfr,Murfreesboro Pk,Murfreesboro Pike (40 mph)
3,I-40,I-40,4,I-40,I-40,I-40 (70 mph)


### 4c. Number functions
`ROUND, ABS, CAST`, plus arithmetic `+ - * /`. Note: integer ÷ integer is **integer division** in SQLite, so multiply by `1.0` or `CAST` to get decimals.

In [15]:
sql("""
SELECT month, arrivals, departures,
       arrivals + departures                         AS total_ops,
       ROUND((arrivals + departures) / 31.0, 1)       AS avg_ops_per_day,
       ABS(arrivals - departures)                     AS imbalance,
       CAST(arrivals AS REAL) / departures            AS ratio
FROM airport_ops
""")

,month,arrivals,departures,total_ops,avg_ops_per_day,imbalance,ratio
0,2026-01,11800,11750,23550,759.7,50,1.004255
1,2026-02,11200,11300,22500,725.8,100,0.991150
2,2026-03,13100,13050,26150,843.5,50,1.003831


### 4d. Date and time functions
SQLite stores dates as text and uses `strftime(format, value)`. MySQL equivalents are in the table at the end.
| Need | SQLite | MySQL |
|---|---|---|
| date part | `date(ts)` | `DATE(ts)` |
| hour | `strftime('%H', ts)` | `HOUR(ts)` |
| weekday name | `strftime('%w', ts)` (0=Sun) | `DAYNAME(ts)` |
| month key | `strftime('%Y-%m', ts)` | `DATE_FORMAT(ts,'%Y-%m')` |
| text → date | already text in ISO form | `STR_TO_DATE(txt,'%m/%d/%Y')` |

In [16]:
sql("""
SELECT occurred_at,
       date(occurred_at)                         AS day,
       CAST(strftime('%H', occurred_at) AS INTEGER) AS hour,
       strftime('%w', occurred_at)               AS weekday_num,
       strftime('%Y-%m', occurred_at)            AS month_key
FROM incidents LIMIT 5
""")

,occurred_at,day,hour,weekday_num,month_key
0,2026-01-01 17:35:00,2026-01-01,17,4,2026-01
1,2026-01-03 16:46:00,2026-01-03,16,6,2026-01
2,2026-01-04 07:00:00,2026-01-04,7,0,2026-01
3,2026-01-04 16:09:00,2026-01-04,16,0,2026-01
4,2026-01-04 17:10:00,2026-01-04,17,0,2026-01


### 4e. NULL handling and CASE
`COALESCE(a, b)` returns the first non-NULL value (MySQL also has `IFNULL`). `CASE WHEN` is SQL's if/else, great for flags and categories.

In [17]:
sql("""
SELECT day, prcp, tmax,
       COALESCE(prcp, 0) AS prcp_clean,
       CASE WHEN prcp > 0 THEN 'Rain' ELSE 'No rain' END AS weather_state,
       CASE WHEN tmax < 40 THEN 'Cold'
            WHEN tmax < 60 THEN 'Cool'
            ELSE 'Mild' END                            AS temp_band
FROM weather LIMIT 6
""")

,day,prcp,tmax,prcp_clean,weather_state,temp_band
0,2026-01-01,0.00,42,0.00,No rain,Cool
1,2026-01-02,0.00,45,0.00,No rain,Cool
2,2026-01-03,0.00,41,0.00,No rain,Cool
3,2026-01-04,0.37,63,0.37,Rain,Mild
4,2026-01-05,0.00,37,0.00,No rain,Cold
5,2026-01-06,0.00,50,0.00,No rain,Cool


## 5. GROUP BY and HAVING
`GROUP BY` makes one row per group; combine it with aggregates.
`WHERE` filters **rows before** grouping, `HAVING` filters **groups after** grouping.

In [18]:
display(sql("""
SELECT incident_type, COUNT(*) AS n, SUM(injuries) AS injuries
FROM incidents
GROUP BY incident_type
ORDER BY n DESC
"""))

,incident_type,n,injuries
0,Stalled vehicle,42,27
1,Debris,42,28
2,Crash,42,29
3,Minor collision,41,22


In [19]:
# Incidents by hour of day -> where does congestion peak?
sql("""
SELECT CAST(strftime('%H', occurred_at) AS INTEGER) AS hour, COUNT(*) AS incidents
FROM incidents
GROUP BY hour
ORDER BY incidents DESC
LIMIT 6
""")

,hour,incidents
0,17,44
1,18,27
2,16,23
3,9,23
4,7,23
5,8,17


In [20]:
# HAVING: keep only groups that pass a test
sql("""
SELECT strftime('%Y-%m', occurred_at) AS month, COUNT(*) AS incidents
FROM incidents
GROUP BY month
HAVING COUNT(*) > 50              -- if you get no rows, lower this number
ORDER BY month
""")

,month,incidents
0,2026-01,54
1,2026-02,65


## 6. JOINs
A join combines rows from two tables using a matching column.
| Join | Keeps |
|---|---|
| `INNER JOIN` | only rows that match in both |
| `LEFT JOIN` | all left rows, matches from the right (NULL if none) |
| `RIGHT JOIN` | all right rows (swap the tables and use LEFT in older SQLite) |
| `FULL OUTER JOIN` | all rows from both |
| `CROSS JOIN` | every combination |
| self join | a table joined to itself |

In [21]:
# INNER JOIN: incidents with the road name
sql("""
SELECT i.incident_id, i.occurred_at, r.road_name, i.incident_type
FROM incidents AS i
INNER JOIN road AS r ON i.road_id = r.road_id
LIMIT 5
""")

,incident_id,occurred_at,road_name,incident_type
0,1,2026-01-01 17:35:00,Murfreesboro Pike,Minor collision
1,2,2026-01-03 16:46:00,Murfreesboro Pike,Debris
2,3,2026-01-04 18:02:00,I-40,Minor collision
3,4,2026-01-04 07:00:00,Briley Parkway,Stalled vehicle
4,5,2026-01-04 17:10:00,Donelson Pike,Minor collision


In [22]:
# JOIN + GROUP BY: incidents per road
sql("""
SELECT r.road_name, COUNT(*) AS incidents, ROUND(AVG(i.injuries), 2) AS avg_injuries
FROM incidents i
JOIN road r ON r.road_id = i.road_id
GROUP BY r.road_name
ORDER BY incidents DESC
""")

,road_name,incidents,avg_injuries
0,I-40,45,0.53
1,Donelson Pike,43,0.67
2,Murfreesboro Pike,42,0.60
3,Briley Parkway,37,0.76


In [23]:
# LEFT JOIN: keep ALL weather days, even days with zero incidents. This matters for your project!
sql("""
SELECT w.day, w.prcp, COUNT(i.incident_id) AS incidents      -- COUNT(column) ignores NULLs, so unmatched days give 0
FROM weather w
LEFT JOIN incidents i ON date(i.occurred_at) = w.day
GROUP BY w.day, w.prcp
ORDER BY incidents ASC
LIMIT 5
""")

,day,prcp,incidents
0,2026-01-02,0.0,0
1,2026-01-06,0.0,0
2,2026-01-10,0.0,0
3,2026-01-11,0.0,0
4,2026-01-12,0.0,0


In [24]:
# RIGHT / FULL joins need SQLite 3.39+. Colab may have an older version, so we guard the call.
try:
    display(sql("SELECT r.road_name, i.incident_id FROM incidents i RIGHT JOIN road r ON i.road_id = r.road_id LIMIT 3"))
except Exception as e:
    print("RIGHT JOIN not supported here:", e)
    print("Equivalent: put the table you want to keep FIRST and use LEFT JOIN.")

,road_name,incident_id
0,Murfreesboro Pike,1
1,Murfreesboro Pike,2
2,I-40,3


In [25]:
# CROSS JOIN: every road x every type (useful to build a complete grid)
sql("SELECT COUNT(*) AS combos FROM road CROSS JOIN (SELECT DISTINCT incident_type FROM incidents) AS t")

,combos
0,16


In [26]:
# SELF JOIN: pair each road with other roads that have a higher speed limit
sql("""
SELECT a.road_name AS slower, b.road_name AS faster
FROM road a JOIN road b ON a.speed_limit < b.speed_limit
ORDER BY a.road_name
LIMIT 6
""")

,slower,faster
0,Briley Parkway,I-40
1,Donelson Pike,Briley Parkway
2,Donelson Pike,I-40
3,Murfreesboro Pike,Donelson Pike
4,Murfreesboro Pike,Briley Parkway
5,Murfreesboro Pike,I-40


### The fan-out trap (important for your project)
Joining tables at **different grains** repeats rows and inflates totals. Below, each incident is joined to its month's flight total (one row per incident) and the flights are then summed, so each month's flights are counted once **per incident** instead of once per month.

In [27]:
# WRONG: monthly flights repeated for every incident row
bad = sql("""
SELECT SUM(a.arrivals + a.departures) AS inflated_total
FROM incidents i JOIN airport_ops a ON strftime('%Y-%m', i.occurred_at) = a.month
""")
good = sql("SELECT SUM(arrivals + departures) AS true_total FROM airport_ops")
display(bad); display(good)

,inflated_total
0,3989400


,true_total
0,72200


**Fix:** aggregate each table to the same grain (here: day or month) *before* joining.

In [28]:
# Monthly grain: incidents per month joined to flights per month (one row each, no fan-out)
sql("""
SELECT a.month,
       a.arrivals + a.departures AS total_ops,
       COALESCE(m.incidents, 0)  AS incidents
FROM airport_ops a
LEFT JOIN (SELECT strftime('%Y-%m', occurred_at) AS month, COUNT(*) AS incidents
           FROM incidents GROUP BY 1) m ON m.month = a.month
ORDER BY a.month
""")

,month,total_ops,incidents
0,2026-01,23550,54
1,2026-02,22500,65
2,2026-03,26150,48


## 7. Subqueries, CTEs, Views
- **Subquery:** a query inside another query.
- **CTE** (`WITH name AS (...)`): a named, readable step.
- **View:** a saved query that behaves like a table.

In [29]:
# Subquery in WHERE: days wetter than average
sql("""
SELECT day, prcp FROM weather
WHERE prcp > (SELECT AVG(prcp) FROM weather)
ORDER BY prcp DESC LIMIT 5
""")

,day,prcp
0,2026-02-06,1.16
1,2026-03-22,1.10
2,2026-02-25,1.03
3,2026-02-21,1.00
4,2026-02-07,0.93


In [30]:
# CTE: build the daily table in readable steps
sql("""
WITH daily AS (
    SELECT date(occurred_at) AS day, COUNT(*) AS incidents
    FROM incidents GROUP BY 1
)
SELECT w.day, w.prcp, COALESCE(d.incidents, 0) AS incidents
FROM weather w LEFT JOIN daily d ON d.day = w.day
ORDER BY incidents DESC LIMIT 5
""")

,day,prcp,incidents
0,2026-01-19,0.28,5
1,2026-01-26,0.68,5
2,2026-02-06,1.16,5
3,2026-02-14,0.28,5
4,2026-03-02,0.12,5


In [31]:
# VIEW: save the integrated daily table once, query it many times
sql("DROP VIEW IF EXISTS v_daily")
sql("""
CREATE VIEW v_daily AS
SELECT w.day,
       w.prcp,
       w.tmax,
       CASE WHEN w.prcp > 0 THEN 'Rain' ELSE 'No rain' END AS weather_state,
       COALESCE(d.incidents, 0) AS incidents,
       a.arrivals + a.departures AS monthly_ops
FROM weather w
LEFT JOIN (SELECT date(occurred_at) AS day, COUNT(*) AS incidents FROM incidents GROUP BY 1) d ON d.day = w.day
LEFT JOIN airport_ops a ON a.month = strftime('%Y-%m', w.day)
""")
sql("SELECT * FROM v_daily LIMIT 5")

OK
OK


,day,prcp,tmax,weather_state,incidents,monthly_ops
0,2026-01-01,0.00,42,No rain,1,23550
1,2026-01-02,0.00,45,No rain,0,23550
2,2026-01-03,0.00,41,No rain,1,23550
3,2026-01-04,0.37,63,Rain,4,23550
4,2026-01-05,0.00,37,No rain,1,23550


In [32]:
# Use the view: average incidents, rain vs no rain (a core question of the project)
sql("""
SELECT weather_state, COUNT(*) AS days, ROUND(AVG(incidents), 2) AS avg_incidents, MAX(incidents) AS worst_day
FROM v_daily
GROUP BY weather_state
""")

,weather_state,days,avg_incidents,worst_day
0,No rain,63,1.05,2
1,Rain,27,3.74,5


## 8. Window functions
They calculate across related rows **without collapsing them**. Syntax: `function() OVER (PARTITION BY ... ORDER BY ...)`.
(Needs SQLite 3.25+, which Colab has; MySQL 8 supports them too.)

In [33]:
sql("""
SELECT day, incidents,
       ROW_NUMBER() OVER (ORDER BY incidents DESC)               AS rank_overall,
       SUM(incidents) OVER (ORDER BY day)                        AS running_total,
       ROUND(AVG(incidents) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS avg_7day
FROM v_daily
ORDER BY day
LIMIT 10
""")

,day,incidents,rank_overall,running_total,avg_7day
0,2026-01-01,1,51,1,1.00
1,2026-01-02,0,71,1,0.50
2,2026-01-03,1,52,2,0.67
3,2026-01-04,4,6,6,1.50
4,2026-01-05,1,53,7,1.40
5,2026-01-06,0,72,7,1.17
6,2026-01-07,2,28,9,1.29
7,2026-01-08,3,16,12,1.57
8,2026-01-09,1,54,13,1.71
9,2026-01-10,0,73,13,1.57


## 9. Set operations
`UNION` (combine, remove duplicates), `UNION ALL` (combine, keep all), `INTERSECT` (in both), `EXCEPT` (in first only).

In [34]:
sql("""
SELECT day FROM v_daily WHERE weather_state = 'Rain'
INTERSECT
SELECT day FROM v_daily WHERE incidents >= 5
ORDER BY day LIMIT 5
""")

,day
0,2026-01-19
1,2026-01-26
2,2026-02-06
3,2026-02-14
4,2026-03-02


## 10. Transactions
A transaction groups changes so they succeed or fail **together**. `BEGIN`, then `COMMIT` to keep or `ROLLBACK` to undo. Use this to test risky updates.

In [35]:
conn.execute("BEGIN")
conn.execute("UPDATE incidents SET injuries = 99")                      # oops: no WHERE
print("Inside transaction:", conn.execute("SELECT MAX(injuries) FROM incidents").fetchone())
conn.execute("ROLLBACK")                                                # undo it
print("After rollback:   ", conn.execute("SELECT MAX(injuries) FROM incidents").fetchone())

Inside transaction: (99,)
After rollback:    (2,)


## 11. SQLite vs MySQL cheat sheet
Your course uses MySQL Workbench. The logic is the same; only some names differ.
| Task | This notebook (SQLite) | MySQL |
|---|---|---|
| show tables | `SELECT name FROM sqlite_master...` | `SHOW TABLES;` |
| describe table | `PRAGMA table_info(t)` | `DESCRIBE t;` |
| auto id | `INTEGER PRIMARY KEY` | `INT AUTO_INCREMENT PRIMARY KEY` |
| text type | `TEXT` | `VARCHAR(n)` |
| decimal | `REAL` | `DECIMAL(p,s)` |
| join text | `a \|\| b` | `CONCAT(a, b)` |
| hour of datetime | `strftime('%H', ts)` | `HOUR(ts)` |
| date of datetime | `date(ts)` | `DATE(ts)` |
| text → date | (store ISO text) | `STR_TO_DATE(txt, '%m/%d/%Y')` |
| first N rows | `LIMIT n` | `LIMIT n` |
| if/else | `CASE WHEN` | `CASE WHEN` or `IF()` |
| group by alias | allowed | allowed |
| FULL OUTER JOIN | 3.39+ | not supported (use `UNION` of LEFT and RIGHT) |
| replace a view | `DROP VIEW IF EXISTS v; CREATE VIEW ...` | `CREATE OR REPLACE VIEW v AS ...` |
| INTERSECT / EXCEPT | supported | MySQL 8.0.31+ only |
| `\|\|` operator | joins text | means OR unless `PIPES_AS_CONCAT` is set; use `CONCAT()` |
| subquery in FROM | alias optional | alias **required** |

## 12. Practice (write your own queries)
Each cell is empty. Write the query, run it, and compare with the hint. Answers are in the *Solutions* section at the bottom, so try first.

1. List the 10 most recent incidents.
2. Count incidents per road name, highest first.
3. Find the average number of injuries for each incident type; keep types averaging above 0.5.
4. Show the 5 days with the most incidents and whether it rained.
5. Add a column `rush_hour` to the incidents query: 'Yes' for hours 7-9 and 16-18, otherwise 'No'; count each.
6. For each month, show total flights and incidents per 1,000 flights.
7. Using `v_daily`, compare average incidents on days above vs below the median monthly operations.

In [36]:
# 1

In [37]:
# 2

In [38]:
# 3

In [39]:
# 4

In [40]:
# 5

In [41]:
# 6

In [42]:
# 7

## 13. Mini project path for BNA (no answers, just the route)
Use the same pattern on your real CSVs:
1. `pd.read_csv` each file → `to_sql` into **raw_** tables (or use Workbench's import wizard).
2. Check: `COUNT(*)`, `NULL` counts, duplicates via `GROUP BY ... HAVING COUNT(*) > 1`.
3. Convert text dates to real dates (`STR_TO_DATE` in MySQL).
4. Reduce each table to **one grain** (day) before joining; start the join from the weather days; `COALESCE` the counts.
5. Save as a **view**, then answer questions with `GROUP BY` and `CASE`.
6. Export results to CSV and chart in Excel.

## Solutions (try first!)
Your numbers may differ slightly from any figures you see elsewhere, because the sample data is random (but fixed by a seed).

In [43]:
# 1
display(sql("SELECT * FROM incidents ORDER BY occurred_at DESC LIMIT 10"))
# 2
display(sql("SELECT r.road_name, COUNT(*) n FROM incidents i JOIN road r USING(road_id) GROUP BY r.road_name ORDER BY n DESC"))
# 3
display(sql("SELECT incident_type, ROUND(AVG(injuries),2) avg_inj FROM incidents GROUP BY incident_type HAVING AVG(injuries) > 0.5"))
# 4
display(sql("SELECT day, incidents, weather_state FROM v_daily ORDER BY incidents DESC LIMIT 5"))
# 5
display(sql("""
SELECT CASE WHEN CAST(strftime('%H',occurred_at) AS INTEGER) IN (7,8,9,16,17,18) THEN 'Yes' ELSE 'No' END AS rush_hour,
       COUNT(*) AS n
FROM incidents GROUP BY rush_hour"""))
# 6
display(sql("""
SELECT a.month, a.arrivals + a.departures AS ops,
       COUNT(i.incident_id) AS incidents,
       ROUND(1000.0 * COUNT(i.incident_id) / (a.arrivals + a.departures), 2) AS per_1000_ops
FROM airport_ops a LEFT JOIN incidents i ON strftime('%Y-%m', i.occurred_at) = a.month
GROUP BY a.month"""))
# 7
display(sql("""
SELECT CASE WHEN monthly_ops > (SELECT AVG(monthly_ops) FROM v_daily) THEN 'High ops' ELSE 'Lower ops' END AS ops_level,
       COUNT(*) AS days, ROUND(AVG(incidents), 2) AS avg_incidents
FROM v_daily GROUP BY ops_level"""))

,incident_id,occurred_at,road_id,incident_type,injuries,severity
0,166,2026-03-30 16:04:00,4,Stalled vehicle,1,High
1,167,2026-03-30 09:17:00,3,Debris,0,Low
2,165,2026-03-28 17:26:00,2,Minor collision,1,High
3,164,2026-03-28 07:52:00,1,Debris,0,Low
4,162,2026-03-27 17:50:00,2,Stalled vehicle,2,High
5,163,2026-03-27 09:21:00,2,Debris,0,Low
6,161,2026-03-26 04:53:00,1,Minor collision,1,High
7,159,2026-03-24 19:03:00,4,Stalled vehicle,0,Low
8,160,2026-03-24 17:50:00,1,Debris,1,High
9,157,2026-03-24 17:04:00,3,Stalled vehicle,1,High


,road_name,n
0,I-40,45
1,Donelson Pike,43
2,Murfreesboro Pike,42
3,Briley Parkway,37


,incident_type,avg_inj
0,Crash,0.69
1,Debris,0.67
2,Minor collision,0.54
3,Stalled vehicle,0.64


,day,incidents,weather_state
0,2026-01-19,5,Rain
1,2026-01-26,5,Rain
2,2026-02-06,5,Rain
3,2026-02-14,5,Rain
4,2026-03-02,5,Rain


,rush_hour,n
0,No,10
1,Yes,157


,month,ops,incidents,per_1000_ops
0,2026-01,23550,54,2.29
1,2026-02,22500,65,2.89
2,2026-03,26150,48,1.84


,ops_level,days,avg_incidents
0,High ops,31,1.55
1,Lower ops,59,2.02
